# 03 · Train the tiny robot VLM and test it in closed loop

We train `TinyVLM` (~2.4M parameters) with **behaviour cloning**: next-token cross-entropy on the expert's action, given image + prompt. Then we evaluate it two ways:

1. **Open loop:** next-action accuracy on unseen states.
2. **Closed loop:** let the model *drive* the simulated robot for full episodes. This is the metric that matters, because small errors compound.

**Runtime:** about 1-3 min on a desktop NVIDIA GPU. On a CPU expect 15-30 min with the defaults; lower `EPISODES` or `EPOCHS` to go faster.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import time, torch
import matplotlib.pyplot as plt
from edge_vlm.data import generate_samples, action_distribution
from edge_vlm.tokenizer import RobotTokenizer
from edge_vlm.train import build_model, train, get_device, VLMPolicy, action_accuracy, save_checkpoint
from edge_vlm.agent import evaluate_policy, FetchAgent
from edge_vlm.sim import FetchWorld

EPISODES = 3000      # ~14k decision points
EPOCHS = 6
BATCH_SIZE = 64
CKPT = ROOT / 'checkpoints' / 'tiny_robot_vlm.pt'

device = get_device()
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('device:', device)

In [ ]:
t = time.time()
train_samples = generate_samples(EPISODES, seed=0)
val_samples = generate_samples(200, seed=10_000)   # different seed = unseen episodes (no leakage)
print(f'train {len(train_samples)}  val {len(val_samples)}  ({time.time() - t:.0f}s)')
print(action_distribution(train_samples))

tok = RobotTokenizer.from_domain()
model = build_model(tok)
print({k: f'{v/1e6:.2f}M' for k, v in model.param_breakdown().items()})

In [ ]:
hist = train(model, tok, train_samples, val_samples, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=2e-3, device=device)
save_checkpoint(model, tok, CKPT)
print('saved ->', CKPT)

plt.figure(figsize=(6, 3))
plt.plot(hist['step'], hist['train_loss'])
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('train loss'); plt.title('behaviour cloning loss'); plt.show()

## Open-loop accuracy (per skill)

Watch `pick` and the first `navigate` after arriving in a room. Those are the decisions that need the **image**. `return` and `handover` only need the text state, so they are easy.

In [ ]:
policy = VLMPolicy(model, tok, device=device)
acc = action_accuracy(policy, val_samples)
for k, v in sorted(acc.items()):
    print(f'{k:10s} {v:.3f}')

## Ablation: is the model really looking?

Replace every image with a blank gray frame and re-measure. If accuracy on `pick` collapses, the model was using vision (good). If nothing changes, it found a text shortcut (bad).

In [ ]:
import numpy as np
from dataclasses import replace
blind = [replace(s, image=np.full_like(s.image, 128)) for s in val_samples]
acc_blind = action_accuracy(policy, blind)
print(f"{'skill':10s} {'with image':>10s} {'blind':>8s}")
for k in sorted(acc):
    print(f'{k:10s} {acc[k]:10.3f} {acc_blind.get(k, 0):8.3f}')

## Closed loop: let the VLM drive

`FetchAgent` runs observe → VLM → **guardrail** (parse + preconditions) → execute. If the model outputs something malformed or unsafe, the robot stops safely instead of doing something random.

In [ ]:
t = time.time()
stats = evaluate_policy(policy, n_episodes=200, seed=777)
print(stats, f'({time.time() - t:.1f}s)')

world = FetchWorld(seed=2024)
res = FetchAgent(policy).run(world)
print('\ncommand:', world.command, '| target in', world.target_room)
for r in res.trajectory:
    print(f"  {r['room']:12s} holding {str(r['holding']):12s} -> {r['output']:18s} | {r['result']}")
print('success:', res.success)

## Failure analysis

Look at the mistakes. Is it confusing similar shapes (cup vs box), colors, or rooms? The usual fixes are more data or harder distractors for those cases, a higher image resolution, or more visual tokens (lower `pixel_shuffle`).

In [ ]:
errors = [(s, policy.predict(s.image, s.prompt)) for s in val_samples]
errors = [(s, p) for s, p in errors if p != s.action]
print(len(errors), 'errors out of', len(val_samples))
if errors:
    fig, axes = plt.subplots(1, min(6, len(errors)), figsize=(14, 3), squeeze=False)
    for ax, (s, p) in zip(axes[0], errors):
        ax.imshow(s.image); ax.axis('off')
        ax.set_title(f'want: {s.action}\ngot:  {p}', fontsize=8)
    plt.show()

## What you just built

A 2.4M-parameter VLM that grounds a language command in camera pixels and picks robot skills. It is about 100x smaller than SmolVLM-256M and runs in milliseconds even on a CPU.

**Its limit:** it only knows ~60 words. Say *"could you get me my scarlet mug"* and it is lost. Fixing that needs a pretrained language model, which is notebook 04.

### Exercises
1. Train with `noise_prob=0.0` in `generate_samples`. Does closed-loop success drop even if open-loop accuracy looks fine? (compounding errors)
2. Train with only 300 episodes. How does data size affect `pick` accuracy?
3. Try `build_model(tok, pixel_shuffle=1)` (64 visual tokens). Better accuracy? How much slower?